# 读取增强版国家队因子：每个 sheet 与每个标的

- 从 **Data** 目录读取 `因子_国家队_增强版_pair_zscore.xlsx`
- 每个 **sheet** 对应一个因子类型（dt）
- 每个 **标的**（pair）为 sheet 内除「交易日期」外的列名
- 统一日期列名、规整类型，便于后续与 y 对齐

In [1]:
import os
import sys
import pandas as pd

_cwd = os.getcwd()
_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'factor_build' else _cwd
if _root not in sys.path:
    sys.path.insert(0, _root)
import config

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

DATA_DIR = config.get_data_dir()
DATE_COL = config.DATE_COL
FACTOR_FILE = getattr(config, 'AUGMENTED_FACTOR_FILENAME', '因子_国家队_增强版_pair_zscore.xlsx')
factor_path = os.path.join(DATA_DIR, FACTOR_FILE)
print('因子文件:', factor_path)
print('存在:', os.path.isfile(factor_path))

因子文件: /Users/amadeus/Desktop/p3_adjusted_program/Data/因子_国家队_增强版_pair_zscore.xlsx
存在: True


## 1. 读取所有 sheet

In [2]:
sheets_raw = pd.read_excel(factor_path, sheet_name=None)
sheet_names = [n for n in sheets_raw.keys() if n != 'Target']  # 可选：排除 Target
print('Sheet 数量:', len(sheets_raw))
print('Sheet 列表:', sheet_names)

Sheet 数量: 12
Sheet 列表: ['pct_chg', 'unit_total', 'netinflow', 'NAV_iopv_discount', 'iopv', 'turn', 'amt_btin', 'amt', 'volume', 'volume_btin', 'ETF申购赎回现金差额']


## 2. 规整每个 sheet：统一日期列、列出标的

In [3]:
def normalize_date_col(df):
    """统一日期列名为 config.DATE_COL，并转为 date 类型。"""
    df = df.copy()
    if 'Date' in df.columns and DATE_COL not in df.columns:
        df = df.rename(columns={'Date': DATE_COL})
    if DATE_COL in df.columns:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce').dt.date
    return df


def get_pair_cols(df):
    """标的 = 除日期列外的所有列。"""
    return [c for c in df.columns if c != DATE_COL]


# 规整后的每个 sheet：{ sheet_name: DataFrame }
augmented_sheets = {}
# 每个 (sheet, 标的) 的列表，便于按因子×标的遍历
sheet_pair_list = []

for name in sheet_names:
    df = sheets_raw[name].copy()
    df = normalize_date_col(df)
    augmented_sheets[name] = df
    pairs = get_pair_cols(df)
    for p in pairs:
        sheet_pair_list.append((name, p))

print('规整后 sheet 数:', len(augmented_sheets))
print('(sheet, 标的) 组合数:', len(sheet_pair_list))

规整后 sheet 数: 11
(sheet, 标的) 组合数: 80


## 3. 每个 sheet 的标的一览

In [4]:
summary = []
for name in sheet_names:
    df = augmented_sheets[name]
    pairs = get_pair_cols(df)
    summary.append({
        'sheet': name,
        '标的数': len(pairs),
        '行数': len(df),
        '标的列表': pairs
    })

for s in summary:
    print(f"Sheet: {s['sheet']}  标的数: {s['标的数']}  行数: {s['行数']}")
    print(f"  标的: {s['标的列表'][:8]}{'...' if len(s['标的列表']) > 8 else ''}")
    print()

Sheet: pct_chg  标的数: 10  行数: 1689
  标的: ['159977.SZ', '159952.SZ', '159915.SZ', '588050.SH', '588080.SH', '510050.SH', '510100.SH', 'Unnamed: 8']...

Sheet: unit_total  标的数: 7  行数: 1723
  标的: ['510100.SH', '588050.SH', '159977.SZ', '588080.SH', '510050.SH', '159915.SZ', '159952.SZ']

Sheet: netinflow  标的数: 7  行数: 1689
  标的: ['159977.SZ', '159952.SZ', '159915.SZ', '588050.SH', '588080.SH', '510050.SH', '510100.SH']

Sheet: NAV_iopv_discount  标的数: 7  行数: 1689
  标的: ['159977.SZ', '159952.SZ', '159915.SZ', '588050.SH', '588080.SH', '510050.SH', '510100.SH']

Sheet: iopv  标的数: 7  行数: 1689
  标的: ['159977.SZ', '159952.SZ', '159915.SZ', '588050.SH', '588080.SH', '510050.SH', '510100.SH']

Sheet: turn  标的数: 7  行数: 1689
  标的: ['159977.SZ', '159952.SZ', '159915.SZ', '588050.SH', '588080.SH', '510050.SH', '510100.SH']

Sheet: amt_btin  标的数: 7  行数: 1689
  标的: ['159977.SZ', '159952.SZ', '159915.SZ', '588050.SH', '588080.SH', '510050.SH', '510100.SH']

Sheet: amt  标的数: 7  行数: 1689
  标的: ['159977.SZ',

## 4. 按 sheet × 标的 取单列序列（示例）

In [5]:
def get_factor_series(sheet_name, pair, sheets=None):
    """取指定 sheet、指定标的的 [交易日期, 值] 序列。"""
    sheets = sheets or augmented_sheets
    if sheet_name not in sheets:
        return None
    df = sheets[sheet_name][[DATE_COL, pair]].copy()
    df = df.rename(columns={pair: 'value'})
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    return df.dropna(subset=[DATE_COL, 'value']).sort_values(DATE_COL).reset_index(drop=True)


# 示例：取第一个 sheet 的第一个标的
if sheet_pair_list:
    first_sheet, first_pair = sheet_pair_list[0]
    series = get_factor_series(first_sheet, first_pair)
    print(f'示例: sheet={first_sheet}, 标的={first_pair}')
    print('行数:', len(series))
    display(series.head(10))

示例: sheet=pct_chg, 标的=159977.SZ
行数: 1506


,交易日期,value
0,2019-09-30,-1.272727
1,2019-10-08,-0.736648
2,2019-10-09,0.247372
3,2019-10-10,2.837754
4,2019-10-11,-0.059988
5,2019-10-14,0.720288
6,2019-10-15,-1.132300
7,2019-10-16,-0.060277
8,2019-10-17,0.000000
9,2019-10-18,-0.482509


## 6. Rolling z-score（窗口在 config.ROLLING_ZSCORE_WINDOW 中可调）

对每个因子(sheet)的每个标的列做 **rolling z-score**：  
`z = (x - rolling_mean) / rolling_std`，窗口内 std=0 时置为 0。

In [6]:
ROLLING_WINDOW = getattr(config, 'ROLLING_ZSCORE_WINDOW', 180)
MIN_VALID = getattr(config, 'ROLLING_ZSCORE_MIN_VALID_IN_WINDOW', 180)

def rolling_zscore(series, window):
    """Rolling z-score：仅用窗口内有效值（非空）算均值和标准差，当日有效且窗口内至少 MIN_VALID 个有效值则输出 z，保证连续。"""
    x = pd.to_numeric(series, errors='coerce')
    valid = x.notna()
    x_for_roll = x.where(valid)  # 无效日不参与 rolling 的 mean/std
    roll_mean = x_for_roll.rolling(window=window, min_periods=MIN_VALID).mean()
    roll_std = x_for_roll.rolling(window=window, min_periods=MIN_VALID).std()
    z = (x - roll_mean) / roll_std.where(roll_std > 1e-12)
    z = z.where(valid)  # 仅当日有效才输出
    return z

# augmented_sheets_z: 每个 sheet 的 DataFrame 中，标的列已变为 rolling z-score
augmented_sheets_z = {}
for name in sheet_names:
    df = augmented_sheets[name].copy()
    pairs = get_pair_cols(df)
    for col in pairs:
        if col in df.columns:
            df[col] = rolling_zscore(pd.to_numeric(df[col], errors='coerce'), ROLLING_WINDOW)
    augmented_sheets_z[name] = df

print('Rolling 窗口:', ROLLING_WINDOW, '，仅用有效值算 mean/std，窗口内至少', MIN_VALID, '个有效值才输出 z')
print('augmented_sheets_z 已生成，键:', list(augmented_sheets_z.keys()))

Rolling 窗口: 180 ，仅用有效值算 mean/std，窗口内至少 180 个有效值才输出 z
augmented_sheets_z 已生成，键: ['pct_chg', 'unit_total', 'netinflow', 'NAV_iopv_discount', 'iopv', 'turn', 'amt_btin', 'amt', 'volume', 'volume_btin', 'ETF申购赎回现金差额']


## 7. 相对因子：同一因子内按标的对做差（config.RELATIVE_FACTOR_PAIRS）

对每对 (标的A, 标的B)，在同一 sheet 内计算 **rolling_z(标的A) - rolling_z(标的B)**，形成相对资金净流入类因子。  
列名示例：`510050.SH-510100.SH`。

In [7]:
RELATIVE_PAIRS = getattr(config, 'RELATIVE_FACTOR_PAIRS', [
    ("510050.SH", "510100.SH"),
    ("588080.SH", "588050.SH"),
    ("159915.SZ", "159952.SZ"),
    ("159915.SZ", "159977.SZ"),
])

# relative_factors: { sheet_name: DataFrame }，列 = 交易日期 + 各对 "codeA-codeB"
relative_factors = {}
for name in sheet_names:
    df_z = augmented_sheets_z[name]
    out = df_z[[DATE_COL]].copy()
    for code_a, code_b in RELATIVE_PAIRS:
        if code_a in df_z.columns and code_b in df_z.columns:
            col_name = f"{code_a}-{code_b}"
            out[col_name] = (df_z[code_a] - df_z[code_b]).ffill()
    relative_factors[name] = out

print('相对因子对数:', len(RELATIVE_PAIRS))
print('RELATIVE_FACTOR_PAIRS:', RELATIVE_PAIRS)
for name in list(relative_factors.keys())[:3]:
    print(f'  {name}: 列 {list(relative_factors[name].columns)}')

相对因子对数: 4
RELATIVE_FACTOR_PAIRS: [('510050.SH', '510100.SH'), ('588080.SH', '588050.SH'), ('159915.SZ', '159952.SZ'), ('159915.SZ', '159977.SZ')]
  pct_chg: 列 ['交易日期', '510050.SH-510100.SH', '588080.SH-588050.SH', '159915.SZ-159952.SZ', '159915.SZ-159977.SZ']
  unit_total: 列 ['交易日期', '510050.SH-510100.SH', '588080.SH-588050.SH', '159915.SZ-159952.SZ', '159915.SZ-159977.SZ']
  netinflow: 列 ['交易日期', '510050.SH-510100.SH', '588080.SH-588050.SH', '159915.SZ-159952.SZ', '159915.SZ-159977.SZ']


## 7b. 每组因子有效值汇报

相对因子中：做差为 NaN（任一侧 rolling 窗口内有 0 或空）的日已剔除，此处汇报每组 (sheet, 标的对) 的有效值数量。

In [8]:
valid_counts = []
for name in sheet_names:
    df = relative_factors[name]
    pair_cols = [c for c in df.columns if c != DATE_COL]
    for col in pair_cols:
        n_valid = df[col].notna().sum()
        valid_counts.append({'sheet': name, '因子(标的对)': col, '有效值': int(n_valid)})
valid_df = pd.DataFrame(valid_counts)
print('每组因子有效值（相对因子：rolling 窗口内无 0/空 且 做差有效 的交易日数）')
display(valid_df)
print('汇总: 共', len(valid_df), '组因子')

每组因子有效值（相对因子：rolling 窗口内无 0/空 且 做差有效 的交易日数）


,sheet,因子(标的对),有效值
0,pct_chg,510050.SH-510100.SH,1325
1,pct_chg,588080.SH-588050.SH,1056
2,pct_chg,159915.SZ-159952.SZ,1510
3,pct_chg,159915.SZ-159977.SZ,1328
4,unit_total,510050.SH-510100.SH,1377
5,unit_total,588080.SH-588050.SH,1120
6,unit_total,159915.SZ-159952.SZ,1544
7,unit_total,159915.SZ-159977.SZ,1373
8,netinflow,510050.SH-510100.SH,1326
9,netinflow,588080.SH-588050.SH,1057


汇总: 共 44 组因子


## 8. 相对因子汇总（供下游与 y 对齐）

- **augmented_sheets_z**：各 sheet 的 rolling z-score 序列  
- **relative_factors**：各 sheet 的「标的对做差」相对因子，列名如 `510050.SH-510100.SH`

In [9]:
# 示例：netinflow 的 510050.SH-510100.SH 相对因子前几行
if 'netinflow' in relative_factors:
    display(relative_factors['netinflow'].sample(10))

# 输出各 relative 因子时间序列到 factor_build/outputs（每次运行覆盖更新）
_out_dir = os.path.join(os.getcwd(), getattr(config, 'FACTOR_BUILD_OUTPUTS', 'outputs'))
os.makedirs(_out_dir, exist_ok=True)
_path_rf = os.path.join(_out_dir, '02_relative_factors_timeseries.xlsx')
with pd.ExcelWriter(_path_rf, engine='openpyxl') as _w:
    for _name, _df in relative_factors.items():
        _df.to_excel(_w, sheet_name=_name[:31], index=False)
print('已输出:', _path_rf)

,交易日期,510050.SH-510100.SH,588080.SH-588050.SH,159915.SZ-159952.SZ,159915.SZ-159977.SZ
662,2021-09-22,-0.115965,0.126926,0.045797,2.554680
143,2019-08-05,NaN,NaN,NaN,NaN
994,2023-02-09,1.000308,-0.205249,0.774644,0.768738
736,2022-01-12,-0.544941,-0.116152,-0.150527,-2.888271
765,2022-03-01,0.115172,-0.043516,-0.846405,-0.605139
64,2019-04-10,NaN,NaN,NaN,NaN
310,2020-04-14,NaN,NaN,2.181708,NaN
1310,2024-05-30,0.383477,-0.673851,-0.364545,0.162382
1271,2024-03-29,-0.652195,-0.535025,-0.364545,-0.055173
442,2020-10-30,3.159264,NaN,-0.217428,0.386118


已输出: /Users/amadeus/Desktop/p3_adjusted_program/factor_build/outputs/02_relative_factors_timeseries.xlsx


## 9. Relative factors 大汇总

汇报所有收集到的 relative factor 情况：按 sheet × 标的对 列出有效值、日期范围、均值/标准差。

In [10]:
rows = []
for name in sheet_names:
    df = relative_factors[name]
    pair_cols = [c for c in df.columns if c != DATE_COL]
    for col in pair_cols:
        s = df[col].dropna()
        n = len(s)
        if n == 0:
            date_min = date_max = None
            mean_val = std_val = None
        else:
            dates = df.loc[s.index, DATE_COL]
            date_min = dates.min()
            date_max = dates.max()
            mean_val = round(s.mean(), 4)
            std_val = round(s.std(), 4)
        rows.append({
            'sheet(因子类型)': name,
            '标的对': col,
            '有效值': n,
            '日期起': date_min,
            '日期止': date_max,
            'mean': mean_val,
            'std': std_val
        })
summary_all = pd.DataFrame(rows)

print('========== Relative factors 大汇总 ==========')
print('共', len(summary_all), '组 relative factor（sheet × 标的对）')
print('标的对配置:', RELATIVE_PAIRS)
print('Rolling 窗口:', ROLLING_WINDOW)
print()
display(summary_all)
print()
print('按 sheet 汇总有效值:')
by_sheet = summary_all.groupby('sheet(因子类型)')['有效值'].agg(['sum', 'count']).rename(columns={'sum': '总有效值', 'count': '因子数'})
display(by_sheet)
print('总有效观测数（所有因子有效值之和）:', summary_all['有效值'].sum())

========== Relative factors 大汇总 ==========
共 44 组 relative factor（sheet × 标的对）
标的对配置: [('510050.SH', '510100.SH'), ('588080.SH', '588050.SH'), ('159915.SZ', '159952.SZ'), ('159915.SZ', '159977.SZ')]
Rolling 窗口: 180



,sheet(因子类型),标的对,有效值,日期起,日期止,mean,std
0,pct_chg,510050.SH-510100.SH,1325,2020-07-06,2025-12-17,0.0018,0.1303
1,pct_chg,588080.SH-588050.SH,1056,2021-08-10,2025-12-17,-0.0009,0.1297
2,pct_chg,159915.SZ-159952.SZ,1510,2019-09-25,2025-12-17,0.0002,0.1139
3,pct_chg,159915.SZ-159977.SZ,1328,2020-07-01,2025-12-17,0.0008,0.1351
4,unit_total,510050.SH-510100.SH,1377,2020-06-08,2026-02-05,-0.0831,1.7390
5,unit_total,588080.SH-588050.SH,1120,2021-06-29,2026-02-05,0.3625,0.7914
6,unit_total,159915.SZ-159952.SZ,1544,2019-09-25,2026-02-05,-0.3618,0.8830
7,unit_total,159915.SZ-159977.SZ,1373,2020-06-12,2026-02-05,-0.2845,1.1841
8,netinflow,510050.SH-510100.SH,1326,2020-07-03,2025-12-17,0.1011,1.2474
9,netinflow,588080.SH-588050.SH,1057,2021-08-09,2025-12-17,0.0020,1.1470



按 sheet 汇总有效值:


,总有效值,因子数
sheet(因子类型),,
ETF申购赎回现金差额,5222,4
NAV_iopv_discount,5222,4
amt,5222,4
amt_btin,5222,4
iopv,5222,4
netinflow,5222,4
pct_chg,5219,4
turn,5222,4
unit_total,5414,4


总有效观测数（所有因子有效值之和）: 57631


## 5. 汇总变量（供后续 notebook 使用）

- **augmented_sheets**：{ sheet_name: DataFrame }，已规整日期列
- **sheet_pair_list**：[(sheet, 标的), ...] 全部组合
- **get_factor_series(sheet_name, pair)**：取某 sheet 某标的的时序

In [11]:
print('augmented_sheets 键:', list(augmented_sheets.keys()))
print('sheet_pair_list 前 5 个:', sheet_pair_list[:5])

augmented_sheets 键: ['pct_chg', 'unit_total', 'netinflow', 'NAV_iopv_discount', 'iopv', 'turn', 'amt_btin', 'amt', 'volume', 'volume_btin', 'ETF申购赎回现金差额']
sheet_pair_list 前 5 个: [('pct_chg', '159977.SZ'), ('pct_chg', '159952.SZ'), ('pct_chg', '159915.SZ'), ('pct_chg', '588050.SH'), ('pct_chg', '588080.SH')]
